# Synlexa Studio — External GPU Compute Node (Research-Use-Only)

Connect this Google Colab GPU (`T4` / `L4` / `A100`) directly to **Synlexa Studio** (`studio.synlexa.com` or Desktop).

1. Select **Runtime -> Change runtime type -> GPU**.
2. Check the toolboxes you want in Cell 1.
3. Run Cell 2 to start `synlexa-compute` + Cloudflare Quick Tunnel and copy the printed **`synlexa-node://...`** pairing string into Studio.

In [ ]:
#@title 1. Install Selected Open-Source Biochemistry Engines
install_vina_rdkit_pyscf = True #@param {type:"boolean"}
install_openmm = True #@param {type:"boolean"}
install_boltz2 = True #@param {type:"boolean"}
install_chemprop = False #@param {type:"boolean"}

!pip install -q pydantic fastapi uvicorn
if install_vina_rdkit_pyscf:
    !pip install -q rdkit vina meeko pyscf
if install_openmm:
    !pip install -q openmm
if install_boltz2:
    !pip install -q boltz
if install_chemprop:
    !pip install -q chemprop

# Install cloudflared for zero-account HTTPS tunnel
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!nvidia-smi || true

In [ ]:
#@title 2. Start Synlexa Compute Bridge & Print Pairing String
# If running from a cloned synlexa-studio repo, set COMPUTE_DIR accordingly:
import os, sys

REPO_URL = "https://github.com/Synlexa/synlexa-studio.git"  #@param {type:"string"}
if not os.path.exists("/content/synlexa-studio"):
    !git clone --depth 1 {REPO_URL} /content/synlexa-studio || true

COMPUTE_DIR = "/content/synlexa-studio/services/compute-worker"
if os.path.exists(COMPUTE_DIR):
    %cd {COMPUTE_DIR}
    !python3 -m synlexa_compute.server --host 127.0.0.1 --port 4180 --tunnel
else:
    print("Upload services/compute-worker from your synlexa-studio repo to /content/compute-worker and run: python3 -m synlexa_compute.server --port 4180 --tunnel")